<a href="https://colab.research.google.com/github/harshal8704/GenAi-Practicals/blob/main/GenAI_Prac4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Machine Translation: Encoder-Decoder LSTM
We will define the training model and the specific layers needed for inference.

In [ ]:
latent_dim = 128

# Encoder
encoder_inputs = Input(shape=(max_encoder_seq_length,))
encoder_embedding_layer = Embedding(num_encoder_tokens, latent_dim, mask_zero=True)
enc_emb = encoder_embedding_layer(encoder_inputs)
encoder_lstm = LSTM(latent_dim, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(shape=(max_decoder_seq_length - 1,))
decoder_embedding_layer = Embedding(num_decoder_tokens, latent_dim, mask_zero=True)
dec_emb = decoder_embedding_layer(decoder_inputs)
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)
decoder_dense = Dense(num_decoder_tokens, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

# Training Model
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='rmsprop', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_13      │ (None, 10)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_14      │ (None, 12)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_4         │ (None, 10, 128)   │    272,128 │ input_layer_13[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_5         │ (None, 10)        │          0 │ input_layer_13[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_5         │ (None, 12, 128)   │    345,600 │ input_layer_14[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_4 (LSTM)       │ [(None, 128),     │    131,584 │ embedding_4[0][0… │
│                     │ (None, 128),      │            │ not_equal_5[0][0] │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_5 (LSTM)       │ [(None, 12, 128), │    131,584 │ embedding_5[0][0… │
│                     │ (None, 128),      │            │ lstm_4[0][1],     │
│                     │ (None, 128)]      │            │ lstm_4[0][2]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 12, 2700)  │    348,300 │ lstm_5[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,229,196 (4.69 MB)

 Trainable params: 1,229,196 (4.69 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Train the model
model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=64,
    epochs=20,
    validation_data=([val_encoder_input_data, val_decoder_input_data], val_decoder_target_data)
)

Epoch 1/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 18s 341ms/step - accuracy: 0.1343 - loss: 7.2932 - val_accuracy: 0.1480 - val_loss: 6.0040
Epoch 2/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 21s 341ms/step - accuracy: 0.1384 - loss: 5.9560 - val_accuracy: 0.1608 - val_loss: 5.6631
Epoch 3/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 13s 320ms/step - accuracy: 0.1483 - loss: 5.7337 - val_accuracy: 0.1613 - val_loss: 5.4737
Epoch 4/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 13s 330ms/step - accuracy: 0.1568 - loss: 5.6153 - val_accuracy: 0.1613 - val_loss: 5.4122
Epoch 5/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 13s 322ms/step - accuracy: 0.1603 - loss: 5.5246 - val_accuracy: 0.1683 - val_loss: 5.3574
Epoch 6/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 14s 332ms/step - accuracy: 0.1625 - loss: 5.4549 - val_accuracy: 0.1693 - val_loss: 5.3149
Epoch 7/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 13s 314ms/step - accuracy: 0.1625 - loss: 5.3919 - val_accuracy: 0.1731 - val_loss: 5.2059
Epoch 8/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 14s 334ms/step - accuracy: 0.1626 - loss: 5.3420 - val_accu

### Inference Models
We reuse the layers from the training model to perform step-by-step prediction.

In [ ]:
encoder_model = Model(encoder_inputs, encoder_states)

decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_inputs_single = Input(shape=(1,))
# Use the layer instance, not the tensor
dec_emb_inf = decoder_embedding_layer(decoder_inputs_single)

decoder_outputs_inf, state_h_inf, state_c_inf = decoder_lstm(
    dec_emb_inf, initial_state=decoder_states_inputs
)
decoder_states_inf = [state_h_inf, state_c_inf]
decoder_outputs_inf = decoder_dense(decoder_outputs_inf)

decoder_model = Model(
    [decoder_inputs_single] + decoder_states_inputs,
    [decoder_outputs_inf] + decoder_states_inf
)

In [ ]:
import numpy as np

# Reverse-lookup token index to words
reverse_input_word_index = {i: w for w, i in input_tokenizer.word_index.items()}
reverse_target_word_index = {i: w for w, i in target_tokenizer.word_index.items()}

def decode_sequence(input_seq):
    # Encode the input as state vectors.
    states_value = encoder_model.predict(input_seq, verbose=0)

    # Generate empty target sequence of length 1 with the start token.
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = target_tokenizer.word_index['starttoken']

    stop_condition = False
    decoded_sentence = ''

    while not stop_condition:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value, verbose=0)

        # Sample a token (greedy search)
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = reverse_target_word_index.get(sampled_token_index, '')

        if sampled_word == 'endtoken' or len(decoded_sentence.split()) > max_decoder_seq_length:
            stop_condition = True
        else:
            decoded_sentence += ' ' + sampled_word

        # Update the target sequence (of length 1).
        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_token_index

        # Update states
        states_value = [h, c]

    return decoded_sentence.strip()

In [ ]:
# Test on a few validation samples
for seq_index in range(5):
    input_seq = val_encoder_input_data[seq_index: seq_index + 1]
    decoded_sentence = decode_sequence(input_seq)
    print('-')
    print('Input sentence:', val_input_texts[seq_index])
    print('Decoded sentence:', decoded_sentence)
    print('Target sentence:', val_target_texts[seq_index].replace('starttoken', '').replace('endtoken', '').strip())

-
Input sentence: i've made a decision.
Decoded sentence: मुझे बहुत है।
Target sentence: मैं फ़ैसला कर चुका हूँ।
-
Input sentence: i chose to leave instead of staying behind.
Decoded sentence: मुझे ने में में लिए नहीं है।
Target sentence: मैंने रहने की बजाय जाने का इरादा किया।
-
Input sentence: this is my third marriage.
Decoded sentence: मुझे में है।
Target sentence: यह मेरी तीसरी शादी है।
-
Input sentence: i get up at six every day.
Decoded sentence: मुझे ने में में लिए नहीं है।
Target sentence: मैं रोज़ छः बजे उठता हूँ।
-
Input sentence: it's like a dream come true.
Decoded sentence: मुझे ने में लिए है।
Target sentence: मानो कोई सपना सच आ गया हो।


In [ ]:
def translate_user_input(text):
    # Preprocess
    text = text.lower().strip()
    seq = input_tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=max_encoder_seq_length, padding='post')

    # Decode
    translation = decode_sequence(padded)
    return translation

# Example usage:
test_sentence = "I am a student."
print(f"English: {test_sentence}")
print(f"Hindi: {translate_user_input(test_sentence)}")

English: I am a student.
Hindi: मुझे में है।
